<a href="https://colab.research.google.com/github/beyzadurdu6619/TrustLLM-Uncertainty-Quantification/blob/main/notebooks/07_week.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# TR: Model ve tokenizer yükleme
# EN: Load model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


def generate_responses(prompt, num_samples=10, max_new_tokens=25):
    """TR: Aynı soru için sıcaklık ölçeklemesiyle N adet farklı yanıt üretir.

    EN: Generates N different responses for the same prompt using temperature
    sampling.
    """
    inputs = tokenizer(prompt, return_tensors="pt")
    responses = []

    for _ in range(num_samples):
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
        # TR: Sadece yeni üretilen metni çözüyoruz
        # EN: Decode only the newly generated text
        generated_text = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
        )
        responses.append(generated_text.strip())

    return responses

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

# TR: Metin benzerliğini ölçen embedding modeli
# EN: Embedding model for measuring text similarity
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def cluster_responses(responses, distance_threshold=0.3):
    """TR: Üretilen metin yanıtlarını anlamsal benzerliklerine göre gruplar.

    EN: Groups text responses based on semantic similarity using clustering.
    """
    embeddings = embedder.encode(responses)

    # TR: Kosinüs mesafesine göre anlamsal kümeleme
    # EN: Hierarchical clustering based on cosine distance
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=distance_threshold,
    )

    cluster_labels = clustering.fit_predict(embeddings)
    return cluster_labels

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
import numpy as np


def calculate_semantic_entropy(cluster_labels):
    """TR: Anlamsal kümeler üzerinden Entropi (Belirsizlik) skoru hesaplar.

    EN: Calculates Entropy (Uncertainty) score over semantic clusters.
    """
    _, counts = np.unique(cluster_labels, return_counts=True)
    probabilities = counts / len(cluster_labels)

    # TR: Entropi hesabı (Sıfıra bölünme hatasını önlemek için 1e-12 ekliyoruz)
    # EN: Entropy calculation (Adding 1e-12 to prevent log(0) error)
    semantic_entropy = -np.sum(probabilities * np.log(probabilities + 1e-12))
    return semantic_entropy